# Tyche Data API — quickstart

Pull prices, returns, correlations, and macro series into pandas.

First, mint an API key in the web app (Settings -> API keys). Copy `.env.example` to
`.env` and fill in `TYCHE_API_KEY` (and `TYCHE_BASE_URL` if not running locally on
`http://localhost:8000`).

In [ ]:
import os

from tyche_client import TycheClient
from dotenv import load_dotenv

load_dotenv(override=True)  # reads TYCHE_BASE_URL / TYCHE_API_KEY from a local .env file (see .env.example)
# override=True: without it, editing .env and re-running this cell in an already-running
# kernel silently keeps the OLD value, since load_dotenv() won't clobber an env var that
# a previous run of this same cell already set -- a real bug this caused once, not hypothetical.

client = TycheClient(
    os.getenv("TYCHE_BASE_URL", "http://localhost:8000"),
    os.getenv("TYCHE_API_KEY", "tyk_paste_your_key_here"),
)

## Single ticker price history

In [ ]:
aapl = client.prices("AAPL", start="2024-01-01")
aapl.tail()

In [ ]:
aapl["Close"].plot(title="AAPL close", figsize=(10, 4));

## Price matrix, returns, and correlations

In [ ]:
tickers = ["SPY", "QQQ", "TLT", "GLD"]
prices = client.price_matrix(tickers, start="2023-01-01")
(prices / prices.iloc[0]).plot(title="Normalized prices", figsize=(10, 4));

In [ ]:
corr = client.correlation_matrix(tickers, start="2023-01-01")
corr

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(corr)), corr.columns)
ax.set_yticks(range(len(corr)), corr.index)
fig.colorbar(im);

## Macro / volatility series

In [ ]:
unrate = client.series("fred", "UNRATE")
vix = client.series("vix", "VIX", start="2024-01-01")
vix.plot(title="VIX", figsize=(10, 4));

## Discovery and rolling correlation

In [ ]:
client.search("apple")

In [ ]:
rc = client.rolling_correlation("ticker", "AAPL", "ticker", "MSFT")
rc["correlation"].plot(title="AAPL vs MSFT — 1Y rolling correlation", figsize=(10, 4));